> **Out of date:** Queen Board now talks to `queen serve` (queen-portable) instead of running `infer.py`. Cells 3–5 below use the old setup and won't hook Queen up. They'll be updated once queen-portable is on GitHub.

# Queen Board on Colab

Play against **Queen 4B** and ask it about positions, on a Colab GPU.

**Runtime → Change runtime type → L4 or A100.** Queen needs BF16 (compute capability 8.0+) and a CUDA-13-compatible driver; a T4 will not work.

*Untested on Colab so far — cell 1 tells you whether your runtime can run it before anything big downloads.*

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
import subprocess
cap = float(subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().split()[0])
drv = int(subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader']).decode().split('.')[0])
print('BF16 OK' if cap >= 8.0 else 'NO BF16: switch to an L4 or A100 runtime')
print('Driver', drv, '- CUDA 13 needs roughly 580+;', 'looks OK' if drv >= 580 else 'probably too old for Queen as released')

In [ ]:
# 2. Get this repo (private: add a GitHub token as a Colab secret named GH_TOKEN)
from google.colab import userdata
import os
token = userdata.get('GH_TOKEN')
REPO = 'YOUR_GITHUB_USER/queen-board'  # <- change this
!rm -rf /content/queen-board
!git clone -q https://{token}@github.com/{REPO}.git /content/queen-board
%cd /content/queen-board
# uv is a standalone binary (not pip); everything else goes into venvs
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv venv -q .venv && uv pip install -q --python .venv/bin/python -r requirements.txt
!.venv/bin/python tools/get_stockfish.py

In [ ]:
# 3. Download Queen (~8.4 GiB) — commands from the model card
%cd /content
!uv venv -q --python 3.12.12 queen_hce-4/.venv
!uv pip install -q --python queen_hce-4/.venv/bin/python huggingface-hub==1.27.0
!queen_hce-4/.venv/bin/hf download princeton-nlp/queen_hce-4 --local-dir queen_hce-4
!uv pip sync -q --python queen_hce-4/.venv/bin/python queen_hce-4/requirements-inference.txt

In [ ]:
# 4. Smoke test Queen on the starting position (slow first time)
!cd /content/queen_hce-4 && .venv/bin/python infer.py --fen 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1' --gpu-memory-utilization 0.6 2>/dev/null | tail -40

In [ ]:
# 5. Start the board server in the background
import subprocess, os, time
env = dict(os.environ, QUEEN_DIR='/content/queen_hce-4', QUEEN_CMD='.venv/bin/python infer.py --fen "{fen}" --history {history} --gpu-memory-utilization 0.6')
server = subprocess.Popen(['/content/queen-board/.venv/bin/python','-m','uvicorn','server.app:app','--port','8000'], cwd='/content/queen-board',
                          env=env, stdout=open('/content/server.log','w'), stderr=subprocess.STDOUT)
time.sleep(3); print(open('/content/server.log').read()[-500:])

In [ ]:
# 6. Open the board (new tab works best)
from google.colab import output
output.serve_kernel_port_as_window(8000)

## Ask Queen from Python
The board's API works from here too — handy for testing prompts without the UI.

In [ ]:
# requests is preinstalled in Colab's own Python; nothing new is installed here
import requests
API = 'http://localhost:8000/api'
requests.post(f'{API}/new', json={'mode':'hotseat', 'fen':'r1bqkb1r/pppp1ppp/2n2n2/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR w KQkq - 4 4'})
r = requests.post(f'{API}/analyze', json={'engine':'queen'}, timeout=900).json()
print(r.get('best_san'), r.get('eval_cp'))
print(r.get('text') or r.get('detail'))